# scorpus Demo Walkthrough

This notebook walks through the full demo workflow from HuggingFace download
through canonicalized corpus and pertTF loader.

## Prerequisites

- Installation completed: `pip install -e ".[demo]"`
- Working Python environment with `scorpus` importable

## Download demo data

In [1]:
from pathlib import Path
import sys

repo_root = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "src" / "scorpus").exists()
)
sys.path.insert(0, str(repo_root / "src"))

import requests

demo_root = repo_root / "demo_data"
base_url = "https://huggingface.co/datasets/weililab/perturb-data-lab-demo/resolve/main"
files = [
    "h5ad/demo_marson_d2_rest.h5ad",
    "h5ad/demo_xorion_hct116_dual_guide.h5ad",
    "checksums.txt",
]

for rel in files:
    out = demo_root / rel
    out.parent.mkdir(parents=True, exist_ok=True)
    if out.exists():
        print(f"  {rel}: already exists")
        continue
    response = requests.get(f"{base_url}/{rel}", stream=True)
    response.raise_for_status()
    with open(out, "wb") as fh:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            fh.write(chunk)

for rel in ["h5ad/demo_marson_d2_rest.h5ad", "h5ad/demo_xorion_hct116_dual_guide.h5ad"]:
    f = demo_root / rel
    size_mb = f.stat().st_size / 1e6 if f.exists() else 0
    print(f"  {rel}: {size_mb:.1f} MB {'(ok)' if f.exists() else 'MISSING'}")

  h5ad/demo_marson_d2_rest.h5ad: already exists
  h5ad/demo_xorion_hct116_dual_guide.h5ad: already exists
  checksums.txt: already exists
  h5ad/demo_marson_d2_rest.h5ad: 25.0 MB (ok)
  h5ad/demo_xorion_hct116_dual_guide.h5ad: 32.8 MB (ok)


## Inspect raw h5ad metadata

Inspection reads metadata and samples matrix candidates without loading full
count matrices.

In [2]:
from scorpus.inspectors import inspect_target
from scorpus.inspectors.models import DatasetSummaryDocument, InspectionTarget

review_dir = repo_root / "artifacts" / "review"
review_dir.mkdir(parents=True, exist_ok=True)

datasets = [
    ("marson_d2_rest", str(demo_root / "h5ad" / "demo_marson_d2_rest.h5ad")),
    ("xorion_hct116_dual_guide", str(demo_root / "h5ad" / "demo_xorion_hct116_dual_guide.h5ad")),
]

for ds_id, source_path in datasets:
    artifacts = inspect_target(
        InspectionTarget(dataset_id=ds_id, source_path=source_path, source_release=ds_id),
        review_dir,
    )
    summary = DatasetSummaryDocument.from_yaml_file(artifacts.inspection_summary)
    print(f"\n{ds_id}:")
    print(f"  cells: {summary.dataset.obs_rows}, features: {summary.dataset.var_rows}")
    print(f"  count source: {summary.count_source_decision.selected_candidate}")
    print(f"  readiness: {summary.materialization_readiness}")

[inspect] start marson_d2_rest


[inspect] done marson_d2_rest count=.X readiness=pass

marson_d2_rest:
  cells: 2720, features: 18130
  count source: .X
  readiness: pass
[inspect] start xorion_hct116_dual_guide


[inspect] done xorion_hct116_dual_guide count=.X readiness=pass

xorion_hct116_dual_guide:
  cells: 2720, features: 38606
  count source: .X
  readiness: pass


## Quick peek at raw metadata

In [3]:
import anndata

for ds_id, source_path in datasets:
    adata = anndata.read_h5ad(source_path, backed="r")
    print(f"\n--- {ds_id} ---")
    print(f"Shape: {adata.shape}")
    cols = [c for c in ["guide_id", "guide_type", "perturbed_gene_name",
                        "guide_target", "gene_target", "sample"]
            if c in adata.obs.columns]
    if cols:
        print(adata.obs[cols].head(5).to_string())


--- marson_d2_rest ---
Shape: (2720, 18130)
                                                                        guide_id guide_type perturbed_gene_name
AAACAAGCAGGCCTCTAAGTAGAG-1_CD4i_R1L23_CD4i_R1_D2_Rest_CD4i_R1_Ultima      TPMT-1  targeting                TPMT
AAAGCATGTGACCTGCAGCTGTGA-1_CD4i_R1L23_CD4i_R1_D2_Rest_CD4i_R1_Ultima  CALCOCO1-1  targeting            CALCOCO1
AAAGCCTAGGAAGGTAAAGTAGAG-1_CD4i_R1L23_CD4i_R1_D2_Rest_CD4i_R1_Ultima     SYT11-1  targeting               SYT11
AACCTCATCATTATGCAGCTGTGA-1_CD4i_R1L23_CD4i_R1_D2_Rest_CD4i_R1_Ultima   CYP20A1-2  targeting             CYP20A1
AAGTTCGCAACCTTGTAGCTGTGA-1_CD4i_R1L23_CD4i_R1_D2_Rest_CD4i_R1_Ultima    CDC25A-1  targeting              CDC25A

--- xorion_hct116_dual_guide ---
Shape: (2720, 38606)
                                                                                                                                                                                       guide_target gene_target         sample
AACGC

## Materialize a federated Lance corpus (both datasets at once)

This builds a **federated** Lance corpus: each dataset gets its own isolated
`matrix/` and `meta/` directory under `{corpus}/{dataset_id}/`. Both datasets
are materialized in one Python loop using the materializer API.

In [4]:
from scorpus.materializers import DatasetMaterializer
from scorpus.materializers.models import OutputRoots, CorpusIndexDocument
from scorpus.materializers.paths import resolve_corpus_paths

corpus_root = repo_root / "artifacts" / "demo_corpus"
corpus_root.mkdir(parents=True, exist_ok=True)

for i, (ds_id, source_path) in enumerate(datasets):
    paths = resolve_corpus_paths("federated", corpus_root, ds_id)
    mode = "create" if i == 0 else "append"

    materializer = DatasetMaterializer(
        source_path=source_path,
        inspection_summary_path=str(review_dir / ds_id / "dataset-summary.yaml"),
        output_roots=OutputRoots(
            metadata_root=str(paths.meta_root),
            matrix_root=str(paths.matrix_root),
        ),
        dataset_id=ds_id,
        backend="lance",
        topology="federated",
        corpus_index_path=str(corpus_root / "corpus-index.yaml"),
        corpus_id="demo_corpus",
        register=True,
        mode=mode,
        dataset_index=i,
        global_row_start=0,
    )
    manifest = materializer.materialize()
    print(f"{mode}: {ds_id} -> {manifest.cell_count} cells, {manifest.feature_count} features")

index_doc = CorpusIndexDocument.from_yaml_file(corpus_root / "corpus-index.yaml")
print([d.dataset_id for d in index_doc.datasets])

create: marson_d2_rest -> 2720 cells, 18130 features


append: xorion_hct116_dual_guide -> 2720 cells, 38606 features
['marson_d2_rest', 'xorion_hct116_dual_guide']


## Install reviewed schemas

In [5]:
import shutil

schema_root = repo_root / "examples" / "demo_canonicalization"
for ds_id, _ in datasets:
    source = schema_root / f"{ds_id}.final-schema.yaml"
    target = corpus_root / ds_id / "meta" / "final-schema.yaml"
    shutil.copyfile(source, target)
    print(f"installed {ds_id} -> {target}")

installed marson_d2_rest -> /autofs/projects-t3/lilab/yangqisu/repos/data_perturb_v2/scorpus/artifacts/demo_corpus/marson_d2_rest/meta/final-schema.yaml
installed xorion_hct116_dual_guide -> /autofs/projects-t3/lilab/yangqisu/repos/data_perturb_v2/scorpus/artifacts/demo_corpus/xorion_hct116_dual_guide/meta/final-schema.yaml


## Canonicalize

In [6]:
from scorpus.canonical import run_canonicalization
from scorpus.materializers.models import MaterializationManifest

index_doc = CorpusIndexDocument.from_yaml_file(corpus_root / "corpus-index.yaml")
for ds in index_doc.datasets:
    manifest = MaterializationManifest.from_yaml_file(corpus_root / ds.manifest_path)
    paths = resolve_corpus_paths("federated", corpus_root, ds.dataset_id)

    result = run_canonicalization(
        dataset_id=ds.dataset_id,
        raw_obs_path=corpus_root / manifest.raw_cell_meta_path,
        raw_var_path=corpus_root / manifest.raw_feature_meta_path,
        size_factor_path=corpus_root / manifest.size_factor_parquet_path,
        schema_path=paths.meta_root / "final-schema.yaml",
        output_root=paths.canonical_meta_root,
    )
    print(f"{result.dataset_id}: {result.obs_rows} obs rows, {result.var_rows} var rows")

marson_d2_rest: 2720 obs rows, 18130 var rows


xorion_hct116_dual_guide: 2720 obs rows, 38606 var rows


## Validate and load the corpus

In [7]:
from scorpus.loaders.validation import validate_corpus_structure
from scorpus.loaders import load_corpus

report = validate_corpus_structure(corpus_root)
print(report["status"], report["topology"], report["total_rows"])

corpus = load_corpus(str(corpus_root))
print(f"Datasets: {corpus.dataset_ids}")
print(f"Total cells: {len(corpus.metadata_index)}")
print(f"Global vocab size: {corpus.feature_registry.global_vocab_size}")

success federated 5440
Datasets: ('marson_d2_rest', 'xorion_hct116_dual_guide')
Total cells: 5440
Global vocab size: 38607


## Inspect canonical metadata

In [8]:
import polars as pl

marson_rows = corpus.take_metadata(
    list(range(0, 10)),
    columns=["dataset_id", "perturb_label", "condition",
             "perturb_type", "cell_context", "batch_id"],
)
xorion_rows = corpus.take_metadata(
    list(range(2720, 2730)),
    columns=["dataset_id", "perturb_label", "condition",
             "perturb_type", "cell_context", "batch_id"],
)

print("--- Marson ---")
print(marson_rows)
print("\n--- Xorion ---")
print(xorion_rows)

meta = corpus.metadata_index.df
all_labels = meta.get_column("perturb_label")
ctrl_count = (pl.Series(all_labels) == "ctrl").sum()
total = len(all_labels)
print(f"\nControl rows: {ctrl_count} / {total} "
      f"({100 * ctrl_count / total:.0f}%)")

--- Marson ---
{'dataset_id': ('marson_d2_rest', 'marson_d2_rest', 'marson_d2_rest', 'marson_d2_rest', 'marson_d2_rest', 'marson_d2_rest', 'marson_d2_rest', 'marson_d2_rest', 'marson_d2_rest', 'marson_d2_rest'), 'perturb_label': ('TPMT', 'CALCOCO1', 'SYT11', 'CYP20A1', 'CDC25A', 'LAIR1', 'SMR3B', 'ctrl', 'TPMT', 'PVR'), 'condition': ('TPMT', 'CALCOCO1', 'SYT11', 'CYP20A1', 'CDC25A', 'LAIR1', 'SMR3B', 'ctrl', 'TPMT', 'PVR'), 'perturb_type': ('crispri', 'crispri', 'crispri', 'crispri', 'crispri', 'crispri', 'crispri', 'control', 'crispri', 'crispri'), 'cell_context': ('CD4+ T cells', 'CD4+ T cells', 'CD4+ T cells', 'CD4+ T cells', 'CD4+ T cells', 'CD4+ T cells', 'CD4+ T cells', 'CD4+ T cells', 'CD4+ T cells', 'CD4+ T cells'), 'batch_id': ('CD4i_R1L23', 'CD4i_R1L23', 'CD4i_R1L23', 'CD4i_R1L23', 'CD4i_R1L23', 'CD4i_R1L23', 'CD4i_R1L23', 'CD4i_R1L23', 'CD4i_R1L23', 'CD4i_R1L23')}

--- Xorion ---
{'dataset_id': ('xorion_hct116_dual_guide', 'xorion_hct116_dual_guide', 'xorion_hct116_dual_guid

## pertTF paired loading

The paired loader and perturbation sampler are no longer part of `scorpus`.
They live in pertTF as `perttf.model.corpus_adapter`; use
`perttf.model.corpus_data.produce_corpus_datasets` to build training and
validation data from a loaded corpus.

For a model-independent sparse batch loader, use
`scorpus.loaders.build_loader(corpus, seq_len=..., batch_size=...)`.


## AnnData handoff (Dask-backed, inner join)

In [10]:
adata = corpus.to_anndata_lazy(
    dataset_id=list(corpus.dataset_ids),
    obs_columns=["perturb_label", "condition", "cell_context", "batch_id"],
    chunk_rows=1024,
    var_join="inner",
)

print(f"adata shape: {adata.shape}")
print(f"adata.X type: {type(adata.X)}")
print(f"Intersection features: {adata.n_vars}")
print(f"obs columns: {list(adata.obs.columns)}")

adata shape: (5440, 18129)
adata.X type: <class 'dask.array.core.Array'>
Intersection features: 18129
obs columns: ['global_row_index', 'dataset_id', 'dataset_index', 'local_row_index', 'cell_id', 'perturb_label', 'condition', 'cell_context', 'batch_id']


/autofs/projects-t3/lilab/yangqisu/repos/data_perturb_v2/scorpus/src/scorpus/loaders/corpus_loader.py:448: UserWarning: selected datasets do not share identical ordered canonical_gene_id axes; exporting 18129 intersection genes
  warnings.warn(


## Quick Scanpy smoke

In [11]:
import scanpy as sc

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000)
print(f"HVG selected: {adata.var['highly_variable'].sum()}")

subset = adata[:100, :100].to_memory()
print(f"Subset shape: {subset.shape}")

HVG selected: 2000
Subset shape: (100, 100)
